# Lab: write your own ground connection strategy
The strategies of satComTopology decide which satellite each ground station connects to: `best-angle`, `best-angle-until-disconnection`, `everything-visible`... You can add your own: extend `GroundToSpaceConnectionStrategy` and register it with the `register_ground_to_space_connectivity_strategy` decorator. Its name can then be used in any configuration, like the built-in ones.

In this lab you will:
1. write the **`InefficientStrategy`**: a ground station whose label contains `ascending` only connects to **ascending** satellites (going north), one whose label contains `descending` only to **descending** satellites (going south). Among them, it takes the best elevation angle, and keeps the link until the satellite is no longer visible
2. try it on **Starlink Group 2** (36 planes × 20 satellites, 720 satellites, 70°, 570 km), with two ground stations in **French Guiana**: an *ascending* one in the capital, Cayenne, and a *descending* one in the jungle, at Saül, 175 km away
3. compute the **shortest path** between them at every tick: 175 km on the ground, but how far through space?
4. replay both paths in an **animated Cesium page**, with satellites colored by their orbital phase

The event factory, the real-time graph and the Cesium page come from [shortest_latency_paths_lab.ipynb](shortest_latency_paths_lab.ipynb).

In [ ]:
import math
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
from geopy.distance import geodesic

from sat_com_application.event_bus.bus import EventBus
from sat_com_application.event_bus.events import (
    EventType,
    ISLCreatedEvent,
    ISLUpdatedEvent,
    GSLCreatedEvent,
)
from sat_com_application.event_bus.factories import DefaultISLEventFactory, EventFactories
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty
from sat_com_connectivity.ground_object_to_space_connectivity.base import GroundToSpaceConnectionStrategy
from sat_com_connectivity.ground_object_to_space_connectivity.ground_connection_strategy_registry import (
    GROUND_TO_SPACE_CONNECTION_STRATEGY_REGISTRY,
    register_ground_to_space_connectivity_strategy,
)
from sat_com_connectivity.ground_object_to_space_connectivity.models import (
    ComputedGroundObjectConnectionsResult,
    GroundToSpaceConnection,
    GroundToSpaceConstraints,
)
from sat_com_connectivity.ground_object_to_space_connectivity.utils import (
    get_best_visible_satellite_from_white_list_unordered,
)
from sat_com_connectivity.ground_object_to_space_connectivity.visibility_angle_utils import (
    check_and_update_elevation_if_satellite_is_visible,
    get_visible_satellites,
)
from sat_com_model.constants import MU_KM3_S2, RE_KM
from sat_com_model.models import GroundObject, GroundToSpaceLink, InterSatelliteLink, Satellite

from cesium_graph_visualizer import AnimatedCesiumGraphVisualizer, default_cesium_property
from dynamic_topology_graph import DynamicTopologyGraph

## 1. The strategy
A strategy only **computes** what should change, it never modifies the simulation: `compute_ground_object_connection_strategy(ground_object, satellites)` is called at every tick, for every ground object of the strategy, and returns a `ComputedGroundObjectConnectionsResult`:
- `obsolete_links`: the links to remove
- `new_ground_object_connections`: the satellites to connect to (`GroundToSpaceConnection`), with the distance and elevation of the link
- `updated_links`: not needed here: the simulation already refreshes the distance and elevation of the links we keep

The simulation manager applies the result, and publishes the `GSL_*` events. The library helpers do the geometry:
- `check_and_update_elevation_if_satellite_is_visible(link)`: is the satellite of an existing link still above the minimum elevation? It also refreshes the link elevation and distance
- `get_visible_satellites(satellites, ground_object, state_cache)`: every satellite above the minimum elevation, with its elevation and distance. `self.state_cache` is set by the simulation manager: the satellite positions are computed once per tick for all the ground stations
- `get_best_visible_satellite_from_white_list_unordered(visible, ground_object)`: the highest one, among the shells of `shells_whitelist`

The orbital phase is `satellite.is_ascending()`: the satellite goes north. A ground station whose label has neither word gets no filter: it behaves like `best-angle-until-disconnection`.

`@register_ground_to_space_connectivity_strategy` adds the class to `GROUND_TO_SPACE_CONNECTION_STRATEGY_REGISTRY`, under its `strategy_name`. The simulation manager builds one instance of each registered strategy when it is created, so the class must be registered **before** `load_simulation()`.

In [ ]:
@register_ground_to_space_connectivity_strategy
class InefficientStrategy(GroundToSpaceConnectionStrategy):
    strategy_name = "inefficient"
    description = (
        "Label with 'ascending': best angle among the visible ascending satellites. "
        "Label with 'descending': same with descending satellites. "
        "Keep the link until the satellite is no longer visible."
    )

    @staticmethod
    def wanted_phase(ground_object: GroundObject) -> bool | None:
        """True: ascending satellites only, False: descending only, None: any."""
        label = (ground_object.label or "").lower()
        if "descending" in label:
            return False
        if "ascending" in label:
            return True
        return None

    def compute_ground_object_connection_strategy(
        self,
        ground_object: GroundObject,
        satellites: list[Satellite],
    ) -> ComputedGroundObjectConnectionsResult:
        result = ComputedGroundObjectConnectionsResult()

        # 1. Keep the current link until its satellite is no longer visible.
        kept_links = []
        for link in ground_object.links:
            if not isinstance(link, GroundToSpaceLink):
                continue
            if check_and_update_elevation_if_satellite_is_visible(link=link):
                kept_links.append(link)
            else:
                result.obsolete_links.append(link)
        if kept_links:
            result.obsolete_links.extend(kept_links[1:])  # a single link at a time
            return result

        # 2. No link: the best visible satellite of the wanted orbital phase.
        wanted_phase = self.wanted_phase(ground_object)
        visible_satellites = [
            visible
            for visible in get_visible_satellites(satellites, ground_object, state_cache=self.state_cache)
            if wanted_phase is None or visible.satellite.is_ascending() == wanted_phase
        ]
        best = get_best_visible_satellite_from_white_list_unordered(visible_satellites, ground_object)
        if best is None:
            return result

        result.new_ground_object_connections.append(
            GroundToSpaceConnection(
                ground_object=ground_object,
                satellite=best.satellite,
                ground_to_space_constraints=GroundToSpaceConstraints(
                    distance_km=best.distance_km,
                    elevation_angle_above_horizon=best.elevation_above_horizon,
                ),
            )
        )
        return result


print(f"registered: {GROUND_TO_SPACE_CONNECTION_STRATEGY_REGISTRY['inefficient'].__name__}")
print(f"available strategies: {', '.join(GROUND_TO_SPACE_CONNECTION_STRATEGY_REGISTRY)}")

## 2. Starlink Group 2 and two ground stations in French Guiana
**Starlink Group 2**: Walker Delta, 36 orbital planes × 20 satellites, 70° inclination, 570 km, with the 4 optical terminals and the full-mesh addressing plan of the previous labs. The configuration takes a mean motion, computed from the altitude for a circular orbit (WGS72, the gravity model of SGP4).

Two ground stations, using the `inefficient` strategy: `Ascending French Guiana - Cayenne`, in the capital on the coast, and `Descending French Guiana - Saül jungle`, in the middle of the Amazonian forest. Their labels select the orbital phase.

Settings: 25° minimum elevation (Starlink's), `shells_whitelist` on Starlink G2, one satellite at a time. With a single link, a ground station can never be a relay in the middle of a path: every path goes through space.

We simulate 1 hour, one tick every 30 seconds.

In [ ]:
def optical_terminal(azimuth, descriptor):
    return {
        "azimuth_angle_configuration": azimuth,
        "elevation_angle_configuration": -90.0,
        "descriptor": descriptor,
        "orientation_possibilities": [descriptor],
    }


def mean_revolution_per_day(altitude_km):
    """Circular orbit, WGS72 constants."""
    return math.sqrt(MU_KM3_S2 / (RE_KM + altitude_km) ** 3) * 86400 / (2 * math.pi)


CAYENNE = (4.9372, -52.3260)
SAUL = (3.6228, -53.2069)

STATIONS = {
    "Ascending French Guiana - Cayenne": CAYENNE,
    "Descending French Guiana - Saül jungle": SAUL,
}


lab_config = {
    "simulation_name": "Custom Ground Strategy Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 01:00:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [
        {
            "identifier": "French Guiana Gateways",
            "type": "ground_station",
            "ground_objects": [
                {"id": index, "label": label, "latitude": latitude, "longitude": longitude, "altitude": 50.0}
                for index, (label, (latitude, longitude)) in enumerate(STATIONS.items())
            ],
            "connectivity_properties": {
                "elevation_above_horizon": 25,
                "shells_whitelist": ["Starlink G2"],
                "maximum_connected_satellites": 1,
                "ground_to_space_connections_strategy": InefficientStrategy.strategy_name,
            },
        }
    ],
    "walker_shells": [
        {
            "type": "delta",
            "static_addressing_plan": "FULL_MESH_WALKER_DELTA",
            "satellite_property": {
                "optical_inter_satellite_link_installations": [
                    optical_terminal(0.0, "INTRA_PLANE_FORWARD"),
                    optical_terminal(-180.0, "INTRA_PLANE_BACKWARD"),
                    optical_terminal(90.0, "INTER_PLANE_LEFT"),
                    optical_terminal(-90.0, "INTER_PLANE_RIGHT"),
                ]
            },
            "constellation_property": {
                "identifier": "Starlink G2",
                "amount_of_orbit_plane": 36,
                "amount_of_satellite_per_orbit_plane": 20,
                "inclination": 70.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": mean_revolution_per_day(570),
            },
            "orbital_connectivity_property": {"adjacent_inter_satellite_shifting": 0},
        }
    ],
}

print(f"Starlink G2 at 570 km: {mean_revolution_per_day(570):.3f} revolutions per day")
simulation_properties = SimulationProperty(**lab_config)

## 3. Record the network and keep a graph in real time
Same as [shortest_latency_paths_lab.ipynb](shortest_latency_paths_lab.ipynb): an ISL event factory adds the position of both satellites to `ISL_CREATED` / `ISL_UPDATED`, our handlers record the positions and the lifetime of every link for the Cesium replay, and `DynamicTopologyGraph` keeps an igraph graph of the network in sync with the events.

In [ ]:
class PositionedISLCreatedEvent(ISLCreatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]  # longitude, latitude, altitude (m)
    satellite_b_position: tuple[float, float, float]


class PositionedISLUpdatedEvent(ISLUpdatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]
    satellite_b_position: tuple[float, float, float]


def position_of(topology_object):
    coordinates = topology_object.get_coordinates()
    return coordinates.longitude, coordinates.latitude, coordinates.altitude


class PositionedISLEventFactory(DefaultISLEventFactory):

    @staticmethod
    def _satellites_information(link: InterSatelliteLink) -> dict:
        satellite_a = link.get_satellite_source()
        satellite_b = link.get_satellite_destination()
        return dict(
            satellite_a_name=satellite_a.satellite_name,
            satellite_b_name=satellite_b.satellite_name,
            satellite_a_shell=satellite_a.shell.identifier,
            satellite_b_shell=satellite_b.shell.identifier,
            satellite_a_position=position_of(satellite_a),
            satellite_b_position=position_of(satellite_b),
        )

    def created(self, link: InterSatelliteLink) -> PositionedISLCreatedEvent:
        event = super().created(link)
        return PositionedISLCreatedEvent(**event.model_dump(), **self._satellites_information(link))

    def updated(self, link: InterSatelliteLink) -> PositionedISLUpdatedEvent:
        event = super().updated(link)
        return PositionedISLUpdatedEvent(**event.model_dump(), **self._satellites_information(link))


event_factories = EventFactories(isl_event_factory=PositionedISLEventFactory())

In [ ]:
satellites = {}
open_links = {}
link_lifetimes = {}


def record_positions(event):
    for satellite_id, name, shell, position in (
        (event.satellite_a_id, event.satellite_a_name, event.satellite_a_shell, event.satellite_a_position),
        (event.satellite_b_id, event.satellite_b_name, event.satellite_b_shell, event.satellite_b_position),
    ):
        satellite = satellites.setdefault(satellite_id, {"name": name, "shell": shell, "positions": {}})
        satellite["positions"][event.sim_time] = position


def open_link(link_id, object_a_id, object_b_id, direction, start_time):
    open_links[link_id] = (object_a_id, object_b_id, direction, start_time)


def close_link(link_id, end_time):
    object_a_id, object_b_id, direction, start_time = open_links.pop(link_id)
    pair = tuple(sorted((object_a_id, object_b_id)))
    lifetime = link_lifetimes.setdefault(pair, {"direction": direction, "intervals": []})
    lifetime["intervals"].append((start_time, end_time))


async def on_isl_created(event: PositionedISLCreatedEvent):
    record_positions(event)
    open_link(event.link_id, event.satellite_a_id, event.satellite_b_id, event.direction, event.sim_time)


async def on_isl_updated(event: PositionedISLUpdatedEvent):
    record_positions(event)


async def on_link_removed(event):
    close_link(event.link_id, event.sim_time)


async def on_gsl_created(event: GSLCreatedEvent):
    open_link(event.link_id, event.ground_station_id, event.satellite_id, "ground", event.sim_time)


event_bus = EventBus()
event_bus.subscribe(EventType.ISL_CREATED, on_isl_created)
event_bus.subscribe(EventType.ISL_UPDATED, on_isl_updated)
event_bus.subscribe(EventType.ISL_REMOVED, on_link_removed)
event_bus.subscribe(EventType.GSL_CREATED, on_gsl_created)
event_bus.subscribe(EventType.GSL_REMOVED, on_link_removed)

topology_graph = DynamicTopologyGraph().attach(event_bus)

## 4. Load the simulation
The manager instantiates every registered strategy, ours included, and gives each one the satellite state cache. `run_synchronously` runs the simulation outside Jupyter's event loop, so each handler runs as soon as its event is published (see the note in [optical_terminal_lab.ipynb](optical_terminal_lab.ipynb)).

In [ ]:
def run_synchronously(function, *args, **kwargs):
    """Run `function` outside Jupyter's event loop, so the event bus calls each handler right away."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(function, *args, **kwargs).result()


configuration_manager = BaseConfigurationManager(
    simulation_property=simulation_properties,
    event_bus=event_bus,
    event_factories=event_factories,
)
simulation_manager = run_synchronously(configuration_manager.load_simulation)

ground_stations = {ground_station.label: ground_station for ground_station in simulation_manager.get_ground_stations()}
print(f"{len(simulation_manager.get_satellites())} satellites")
for label, ground_station in ground_stations.items():
    strategy = ground_station.ground_object_domain.ground_to_space_connections_strategy
    print(f"- {label:<40} strategy {strategy}")
print(f"distance on the ground: {geodesic(CAYENNE, SAUL).km:.0f} km")
print(f"graph: {topology_graph.graph.vcount()} vertices, {topology_graph.graph.ecount()} edges")

## 5. The shortest path at every tick
`measure_routes` runs at every tick, after the simulation update. It asks the graph for the shortest path (Dijkstra, weighted by the link distances) and its propagation latency. It also records the orbital phase of the satellite each ground station is connected to, to check the strategy.

In [ ]:
ROUTES = {
    "Inefficient strategy": ("Ascending French Guiana - Cayenne", "Descending French Guiana - Saül jungle", "#e87ba4"),
}

route_history = {route: [] for route in ROUTES}
phase_history = {label: [] for label in STATIONS}


def connected_phase(ground_station):
    satellites_connected = simulation_manager.get_connected_satellites_to_ground_station(ground_station)
    return satellites_connected[0].orbital_phase if satellites_connected else None


def describe(uid):
    if uid in satellites:
        return satellites[uid]["name"].replace("Starlink G2 ", "S")
    return simulation_manager.get_topology_object_from_uniq_id(uid).label.split(" - ")[-1]


def measure_routes():
    sim_time = simulation_manager.time_manager.current_time
    for label, ground_station in ground_stations.items():
        phase_history[label].append(connected_phase(ground_station))
    for route, (source, target, _) in ROUTES.items():
        path, latency_ms = topology_graph.shortest_path_with_latency_ms(
            ground_stations[source].topology_uniq_id, ground_stations[target].topology_uniq_id
        )
        route_history[route].append({"time": sim_time, "path": path, "latency_ms": latency_ms})

    if sim_time.minute % 10 == 0 and sim_time.second == 0:
        for route in ROUTES:
            sample = route_history[route][-1]
            if sample["path"]:
                print(f"[{sim_time:%H:%M}] {sample['latency_ms']:6.2f} ms, {len(sample['path']) - 1:>2} hops: "
                      + " → ".join(describe(uid) for uid in sample["path"]))
            else:
                print(f"[{sim_time:%H:%M}] no path")


measure_routes()  # state right after the load
simulation_manager.time_manager.register_action(measure_routes)

## 6. Run the simulation

In [ ]:
run_synchronously(simulation_manager.time_manager.tick_until_the_end, ticking_time_in_seconds=30)

end_time = simulation_manager.time_manager.current_time
for link_id in list(open_links):
    close_link(link_id, end_time)

## 7. Did the strategy do its job?
The phase of the satellite each ground station was connected to, over all the ticks. The ascending station should only show `ascending`, the descending one only `descending`.

In [ ]:
for label, phases in phase_history.items():
    ascending = phases.count("ascending")
    descending = phases.count("descending")
    disconnected = phases.count(None)
    print(f"{label:<40} ascending {ascending:>3}   descending {descending:>3}   no satellite {disconnected:>3}   (ticks)")

## 8. The animated Cesium page
Same page as in [shortest_latency_paths_lab.ipynb](shortest_latency_paths_lab.ipynb): the shortest path is underlined on the globe, and the latency chart under it follows the clock. Satellites are **blue** while ascending and **orange** while descending (`default_cesium_property()`): watch the ascending station only link to blue satellites, the descending one to orange ones. The ISLs are gray so the paths stand out.

Without a [Cesium Ion token](https://ion.cesium.com/tokens?page=1), the page uses the offline Natural Earth imagery shipped with CesiumJS.

In [ ]:
CESIUM_TOKEN = None  # or "<YOUR_CESIUM_TOKEN>"

graph = nx.Graph()
for satellite_id, satellite in satellites.items():
    graph.add_node(
        satellite_id,
        type="Satellite",
        name=satellite["name"],
        shell=satellite["shell"],
        positions=[(time, *position) for time, position in satellite["positions"].items()],
    )

for label, ground_station in ground_stations.items():
    graph.add_node(
        ground_station.topology_uniq_id,
        type="GroundStation",
        name=label,
        shell="Ground Station",
        static=True,
        positions=[(simulation_manager.time_manager.start_date, *position_of(ground_station))],
    )

for (object_a_id, object_b_id), lifetime in link_lifetimes.items():
    graph.add_edge(object_a_id, object_b_id, direction=lifetime["direction"], intervals=lifetime["intervals"])


def route_text(sample):
    if not sample["path"]:
        return "no path"
    return f"{sample['latency_ms']:.2f} ms · {len(sample['path']) - 1} hops"


highlighted_paths = {
    f"{route} path": {"color": color, "samples": [(sample["time"], sample["path"]) for sample in route_history[route]]}
    for route, (_, _, color) in ROUTES.items()
}
time_series = {
    "title": "Propagation latency of the shortest path, Cayenne → Saül",
    "unit": "ms",
    "series": {
        route: {"color": color, "samples": [(sample["time"], sample["latency_ms"]) for sample in route_history[route]]}
        for route, (_, _, color) in ROUTES.items()
    },
}
live_values = {route: [(sample["time"], route_text(sample)) for sample in route_history[route]] for route in ROUTES}

# Style: the CesiumProperty of satComTopology's CesiumRenderer. Gray ISLs, so the paths stand out.
cesium_property = default_cesium_property()
cesium_property.cesium_renderer_shell_property = cesium_property.cesium_renderer_shell_property.model_copy(
    update={"orbital_inter_satellite_link_color": "LIGHTSLATEGRAY", "adjacent_inter_satellite_link_color": "SLATEGRAY"}
)

visualizer = AnimatedCesiumGraphVisualizer(
    graph,
    start_time=simulation_manager.time_manager.start_date,
    end_time=end_time,
    title="Starlink G2: the inefficient strategy in French Guiana",
    cesium_token=CESIUM_TOKEN,
    live_values=live_values,
    highlighted_paths=highlighted_paths,
    time_series=time_series,
    trail_seconds=0,
    cesium_property=cesium_property,
)
visualizer.generate_html("results/custom_ground_strategy.html")

Open [`results/custom_ground_strategy.html`](results/custom_ground_strategy.html) in your browser and zoom on French Guiana. Things to watch:
- the two stations are 175 km apart: a satellite above French Guiana sees both, so one satellite and two ground links would be enough. But they never share one: one talks to a satellite going north, the other to one going south. In a Walker Delta, an ascending satellite and a descending one above the same place belong to planes on the opposite sides of the Earth: the ISL mesh joins them only far away, near the turning point of the orbits (70° of latitude). The path goes up towards the north or the south and comes back: many hops, thousands of kilometers
- when a station hands over, the path and its latency jump

## 9. Recap

In [ ]:
def summary(route):
    samples = route_history[route]
    reachable = [sample for sample in samples if sample["path"]]
    latencies = [sample["latency_ms"] for sample in reachable]
    hops = [len(sample["path"]) - 1 for sample in reachable]
    print(route)
    print(f"  reachable  {100 * len(reachable) / len(samples):.0f}% of the ticks")
    if reachable:
        print(f"  latency    min {min(latencies):.2f} ms, mean {sum(latencies) / len(latencies):.2f} ms, max {max(latencies):.2f} ms")
        print(f"  hops       min {min(hops)}, mean {sum(hops) / len(hops):.1f}, max {max(hops)}")
    return latencies


latencies = summary("Inefficient strategy")
ground_km = geodesic(CAYENNE, SAUL).km
ground_ms = ground_km / 299792.458 * 1000
print(f"\nLight along the ground, Cayenne → Saül ({ground_km:.0f} km): {ground_ms:.2f} ms")
if latencies:
    mean_ms = sum(latencies) / len(latencies)
    print(f"Through space with the inefficient strategy: {mean_ms:.0f} ms on average, "
          f"about {mean_ms / 1000 * 299792.458:,.0f} km of path for {ground_km:.0f} km on the ground")

## What's next?
- **A smarter strategy**: prefer the satellite with the longest remaining visibility (`estimate_time_until_disconnection` in `sat_com_connectivity.ground_object_to_space_connectivity.connection_duration_utils`) to reduce the handovers.
- **Compare with a sensible strategy**: switch both stations to `best-angle-until-disconnection` and run the lab again. How often do they share a satellite?
- **Choose the strategy per ground station**: here the label decides. You can also read any other attribute of the ground object or of its `ground_object_domain`.
- **Change the latitude**: move both stations near 60° N, close to the turning point of the orbits. Is the inefficient strategy still that bad?
- Compare with [shortest_latency_paths_lab.ipynb](shortest_latency_paths_lab.ipynb): how many hops does a transatlantic path take with a well-behaved strategy?